In [ ]:
# outdated

In [ ]:
import sys
import os
import numpy as np
import scipy as sp
import pandas as pd
import pickle
import h5py
from matplotlib import pyplot as plt
import seaborn as sns

sys.path.append(os.path.join(os.getcwd(), '../'))
from utils import read_utils

In [ ]:
import importlib
importlib.reload(read_utils)

In [ ]:
path = os.path.join(read_utils.get_data_path(), 'fly-2023-schaffer/brain')

filenames = sorted([name.split('.')[0] for name in os.listdir(path) if not os.path.isdir(os.path.join(path, name)) and not name.endswith('.pkl')])

filenames_runflail = sorted([name.split('.')[0] for name in os.listdir(os.path.join(path, 'incomplete/no_pickle/flailing')) if not os.path.isdir(os.path.join(path, name)) and not name.endswith('.pkl')])

In [ ]:
# prep
def highpass_butter(x, fs, cutoff_hz, order=4, axis=0):
    return sp.signal.sosfiltfilt(sp.signal.butter(order, cutoff_hz, btype='highpass', fs=fs, output='sos'), x, axis=axis)

In [ ]:
traces = dict()

for name in filenames:

	data = {'nwb': dict(),
			'pickle': None}
	def load(name, object):
		# print(name)
		if isinstance(object, h5py.Dataset) \
			and name in ('processing/ophys/DfOverF/RoiResponseSeries/data',
						  'processing/ophys/DfOverF/RoiResponseSeries/timestamps'):
			data['nwb'][name.split('/')[-1]] = object[()]

	with h5py.File(os.path.join(path, f'{name}.nwb'), 'r') as file_handle:
		file_handle.visititems(load)

	sr = read_utils.get_sampling_rate(os.path.join(path, name))
	data['nwb']['data'] = highpass_butter(data['nwb']['data'], sr, 1 / 100)

	if os.path.exists(os.path.join(path, f'{name}_dict.pkl')):
		with open(os.path.join(path, f'{name}_dict.pkl'), 'rb') as file:
			data['pickle'] = dict()
			data['pickle']['dFF'] = pickle.load(file)['dFF']

	traces[name] = data

In [ ]:
traces_runflail = dict()

for name in filenames_runflail:

	data = {'nwb': dict(),
			'pickle': None}
	def load(name, object):
		# print(name)
		if isinstance(object, h5py.Dataset) \
			and name in ('processing/ophys/DfOverF/RoiResponseSeries/data',
						  'processing/ophys/DfOverF/RoiResponseSeries/timestamps'):
			data['nwb'][name.split('/')[-1]] = object[()]

	with h5py.File(os.path.join(path, 'incomplete/no_pickle/flailing', f'{name}.nwb'), 'r') as file_handle:
		file_handle.visititems(load)

	sr = read_utils.get_sampling_rate(os.path.join(path, 'incomplete/no_pickle/flailing', name))
	data['nwb']['data'] = highpass_butter(data['nwb']['data'], sr, 1 / 100)

	if os.path.exists(os.path.join(path, f'{name}_dict.pkl')):
		with open(os.path.join(path, f'{name}_dict.pkl'), 'rb') as file:
			data['pickle'] = dict()
			data['pickle']['dFF'] = pickle.load(file)['dFF']

	traces_runflail[name] = data

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
data = traces['2019_07_01_fly2']['nwb']['data']
timestamps = traces['2019_07_01_fly2']['nwb']['timestamps']
ax.imshow(np.clip(data, -0.015, 0.015).T, aspect='auto', cmap='plasma')
xticks = plt.xticks()[0]
plt.xticks(xticks[1:-1], labels=np.array([f'{timestamps[int(round(n))]:.0f}s' for n in xticks[1:-1]]))
plt.title(f'fig 1h: 2019_07_01_fly2')
plt.show()

In [ ]:
# fig, axis = plt.subplots(len(filenames), 1, figsize=(12, 8 * len(filenames)), squeeze=False)
#
# for i, (name, trace) in enumerate(traces.items()):
# 	ax = axis[i, 0]
#
# 	data = trace['nwb']['data']
# 	timestamps = trace['nwb']['timestamps']
#
# 	ax.imshow(np.clip(data, -0.03, 0.03).T, cmap='plasma', aspect='auto')
# 	ax.set_box_aspect(9/16)
#
# 	xticks = ax.get_xticks()
# 	ax.set_xticks(xticks[1:-1])
# 	ax.set_xticklabels(np.array([f'{timestamps[int(round(n))]:.0f}s' for n in xticks[1:-1]]))
#
# 	ax.set_title(name)
#
# plt.tight_layout()

In [ ]:
for i, (name, trace) in enumerate(traces.items()):
	data = trace['nwb']['data']
	timestamps = trace['nwb']['timestamps']

	fig, axis = plt.subplots(1, 1, figsize=(12, 8), squeeze=False)
	ax = axis[0, 0]
	ax.imshow(np.clip(data, -0.015, 0.015).T, cmap='plasma', aspect='auto')
	ax.set_box_aspect(9/16)

	xticks = ax.get_xticks()
	ax.set_xticks(xticks[1:-1])
	ax.set_xticklabels(np.array([f'{timestamps[int(round(n))]:.0f}s' for n in xticks[1:-1]]))

	ax.set_title(f'{str(i)}: {name}')

	plt.savefig(f'../out/plots/heatmaps/hpf/{str(i)}_{name}.png', bbox_inches='tight')

In [ ]:
# fig, axis = plt.subplots(len(filenames_runflail), 1, figsize=(12, 8 * len(filenames_runflail)), squeeze=False)
#
# for i, (name, trace) in enumerate(traces_runflail.items()):
# 	ax = axis[i, 0]
#
# 	data = trace['nwb']['data']
# 	timestamps = trace['nwb']['timestamps']
#
# 	ax.imshow(np.clip(data, -0.03, 0.03).T, cmap='plasma', aspect='auto')
# 	ax.set_box_aspect(9/16)
#
# 	xticks = ax.get_xticks()
# 	ax.set_xticks(xticks[1:-1])
# 	ax.set_xticklabels(np.array([f'{timestamps[int(round(n))]:.0f}s' for n in xticks[1:-1]]))
#
# 	ax.set_title(name)
#
# plt.tight_layout()

In [ ]:
for i, (name, trace) in enumerate(traces_runflail.items()):
	data = trace['nwb']['data']
	timestamps = trace['nwb']['timestamps']

	fig, axis = plt.subplots(1, 1, figsize=(12, 8), squeeze=False)
	ax = axis[0, 0]
	ax.imshow(np.clip(data, -0.015, 0.015).T, cmap='plasma', aspect='auto')
	ax.set_box_aspect(9/16)

	xticks = ax.get_xticks()
	ax.set_xticks(xticks[1:-1])
	ax.set_xticklabels(np.array([f'{timestamps[int(round(n))]:.0f}s' for n in xticks[1:-1]]))

	ax.set_title(f'{str(i)}: {name}')

	plt.savefig(f'../out/plots/heatmaps/flailing/hpf/{str(i)}_{name}_hpf.png', bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
data = traces_runflail['2019_04_18_fly2']['nwb']['data']
timestamps = traces_runflail['2019_04_18_fly2']['nwb']['timestamps']
ax.imshow(np.clip(data, -0.015, 0.015).T, aspect='auto', cmap='plasma')
xticks = plt.xticks()[0]
plt.xticks(xticks[1:-1], labels=np.array([f'{timestamps[int(round(n))]:.0f}s' for n in xticks[1:-1]]))
plt.title(f'fig 1h: 2019_04_18_fly2')
plt.show()

In [ ]:
# for i, (name, trace) in enumerate(traces.items()):
# 	data = trace['pickle']['dFF'].T
# 	timestamps = trace['nwb']['timestamps']
#
# 	fig, axis = plt.subplots(1, 1, figsize=(12, 8), squeeze=False)
# 	ax = axis[0, 0]
# 	ax.imshow(np.clip(data, -0.015, 0.015).T, cmap='plasma', aspect='auto')
# 	ax.set_box_aspect(9/16)
#
# 	xticks = ax.get_xticks()
# 	ax.set_xticks(xticks[1:-1])
# 	ax.set_xticklabels(np.array([f'{timestamps[int(round(n))]:.0f}s' for n in xticks[1:-1]]))
#
# 	ax.set_title(name)